# Etapa 2 — Screening de Selección (`selection_campaign_1200_50`) y Poda de Modelos

Replica `tools/selection_campaign_1200_50.py` e implementa la lógica de **screening cuantitativo**:
1. Entrena todas las arquitecturas candidatas viables con presupuesto intermedio (`1200 Adam + 50 L-BFGS` en `full_campaign`; escala rápida en `demo_fast`) evaluando frente a referencia analítica y **FEM WSL2**.
2. Calcula un puntaje compuesto adimensional (error en $T_{\max}$, `NRMSE`, desbalance energético y robustez en cable frente a `mms_sine`) para filtrar arquitecturas que se estancan (como `M0` sin escala local o variantes con peor frontera de Pareto).
3. Escribe la lista curada de **modelos promovidos** en `campaign_workspace/promoted_models.json` para reducir drásticamente el cómputo de las etapas masivas siguientes.

In [1]:
import json
import os
import sys
import time
from pathlib import Path
import numpy as np

# Localiza la raíz PINN_V4 aunque el cuaderno se ejecute dentro de notebooks_campaign/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks_campaign":
    ROOT = ROOT.parent
SRC = ROOT / "src"
for p in (SRC, ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Modo de ejecución:
# - 'demo_fast': subconjunto acotado de casos y pocos pasos para verificación rápida en minutos
# - 'full_campaign': reproduce todos los casos, semillas (0,1,2) y presupuestos largos de la tesis
RUN_MODE = "demo_fast"

# Paralelismo de campaña:
# - USE_PARALLEL=True  -> fuerza device='cpu' con ParallelRunner multi-proceso
# - USE_PARALLEL=False -> ejecuta secuencialmente en device='auto' (p. ej. XPU nativo sin fallbacks)
USE_PARALLEL = True
MAX_WORKERS = min(12, os.cpu_count() or 1)
DEVICE = "cpu" if USE_PARALLEL else "auto"

CAMP_WS = ROOT / "notebooks_campaign" / "campaign_workspace"
FEM_CACHE = CAMP_WS / "fem_references"
CAMP_WS.mkdir(parents=True, exist_ok=True)
FEM_CACHE.mkdir(parents=True, exist_ok=True)

# Criterios oficiales de calificación de la campaña final de tesis (2026-10-27)
STRICT_CRITERIA = {"error_Tmax_abs_K": 0.05, "nrmse_pct": 0.25, "balance_pct": 0.5}
RELAXED_CRITERIA = {k: 2.0 * v for k, v in STRICT_CRITERIA.items()}

print(f"PINN_V4 raíz: {ROOT} | RUN_MODE={RUN_MODE!r} | USE_PARALLEL={USE_PARALLEL} (workers={MAX_WORKERS}, device={DEVICE!r}) | Workspace campaña: {CAMP_WS}")

PINN_V4 raíz: C:\usr\ths_mia_fiis\tesis_MIA_2026-2_pinn | RUN_MODE='demo_fast' | USE_PARALLEL=True (workers=12, device='cpu') | Workspace campaña: C:\usr\ths_mia_fiis\tesis_MIA_2026-2_pinn\notebooks_campaign\campaign_workspace


In [2]:
import pinn_thermal.models  # noqa: F401
from pinn_thermal.core.artifact import SolutionArtifact
from pinn_thermal.core.manifest import write_json_atomic
from pinn_thermal.core.parallel import ParallelRunner, RunTask
from pinn_thermal.core.problem import CaseCatalog
from pinn_thermal.core.trainer import PINNTrainer, TrainConfig
from pinn_thermal.fem.runner import FEniCSxAdapter

catalog = CaseCatalog(ROOT / "cases")

def load_or_solve_reference(case_id: str, problem, use_fem: bool = True):
    """Obtiene referencia analítica exacta si existe, o carga/resuelve caché FEM P2 en WSL."""
    xy, _, _ = problem.evaluation()
    exact = problem.reference(xy)
    if exact is not None and np.isfinite(exact).all():
        return np.asarray(exact, dtype=float), "analytic_reference"

    fem_dir = FEM_CACHE / case_id
    fem_json = fem_dir / "fem_l1.json"
    if fem_json.is_file() and fem_json.with_suffix(".npz").is_file():
        art = SolutionArtifact.load(fem_json)
        return art.predict(xy)["T"], f"FEM_WSL_cache:{case_id}"

    if use_fem and FEniCSxAdapter().is_available().get("available"):
        art = FEniCSxAdapter().solve(problem, output_dir=fem_dir, level=1)
        return art.predict(xy)["T"], f"FEM_WSL_L1:{case_id}"

    return None, "physics_only_no_reference"

def extract_metrics(result: dict) -> dict:
    ev = result.get("evaluation") or {}
    bal = ev.get("balance") or {}
    return {
        "error_Tmax_abs_K": ev.get("error_Tmax_abs_K"),
        "nrmse_pct": ev.get("nrmse_pct"),
        "balance_pct": bal.get("global_derived_pct"),
        "Tmax_pred_C": ev.get("Tmax_pred_C"),
        "elapsed_s": result.get("elapsed_s"),
    }
CASES_STAGE2 = ("mms_sine", "cable_disk")
CANDIDATE_MODELS_S2 = (
    ("M0", "M1", "M1_EXACT", "CUSPR", "M0R", "M2")
    if RUN_MODE == "demo_fast"
    else ("M0", "M1", "M2", "DCSNN", "M0R", "CUSP", "CUSPR", "FBPINN", "M1_EXACT", "CUSP_EXACT")
)
ADAM_S2 = 400 if RUN_MODE == "demo_fast" else 2000
LBFGS_S2 = 15 if RUN_MODE == "demo_fast" else 50

In [3]:
# stage2_dir = CAMP_WS / "stage2_selection_1200_50"
stage2_dir = CAMP_WS / "stage2_selection_400_15"
stage2_dir.mkdir(parents=True, exist_ok=True)
stage2_summary = {}
tasks_s2 = []

for case_id in CASES_STAGE2:
    prob = catalog.load(case_id)
    ref_vals, ref_lbl = load_or_solve_reference(case_id, prob, use_fem=True)
    for arch in CANDIDATE_MODELS_S2:
        key = f"{case_id}:{arch}"
        run_dir = stage2_dir / case_id / arch
        tasks_s2.append(
            RunTask(
                name=key,
                problem_source=prob.doc,
                architecture=arch,
                output_dir=run_dir,
                config_kwargs={
                    "adam_steps": ADAM_S2,
                    "lbfgs_steps": LBFGS_S2,
                    "lr": 2e-3,
                    "width": 32,
                    "depth": 3,
                    "weights": {"pde": 1.0, "energy": 100.0},
                    "sampling": "mixed",
                    "focused": True,
                    "focus_criterion": "residual",
                    "focus_surfaces": False,
                    "resample_every": 100,
                    "full_audit": True,
                    "device": DEVICE,
                    "dtype": "float32",
                },
                reference=ref_vals,
                reference_label=ref_lbl,
                overwrite=True,
            )
        )

def _on_stage2_done(res: dict) -> None:
    key = res["name"]
    case_id, arch = key.split(":", 1)
    m = extract_metrics(res)
    stage2_summary[key] = {"case": case_id, "architecture": arch, "status": res.get("status"), **m}
    if res.get("error"):
        stage2_summary[key]["error"] = res.get("error")
    write_json_atomic(CAMP_WS / "stage2_selection_summary.json", stage2_summary)
    print(f"[Etapa 2] {key:24s} | dev={res.get('device')} | NRMSE={m['nrmse_pct']} | dTmax={m['error_Tmax_abs_K']} K | bal={m['balance_pct']}%")

runner_s2 = ParallelRunner(max_workers=MAX_WORKERS if USE_PARALLEL else 1, distribute_devices=not USE_PARALLEL)
runner_s2.run(tasks_s2, on_complete=_on_stage2_done)
write_json_atomic(CAMP_WS / "stage2_selection_summary.json", stage2_summary)

[Etapa 2] mms_sine:M1              | dev=cpu | NRMSE=1.6685251087502286 | dTmax=0.5888957002411388 K | bal=0.02250128440624075%
[Etapa 2] mms_sine:M0              | dev=cpu | NRMSE=0.2439745870428308 | dTmax=0.015394113327076298 K | bal=0.012557948714602647%
[Etapa 2] mms_sine:M2              | dev=cpu | NRMSE=24.828683677606744 | dTmax=12.282911203170826 K | bal=52.32758904603789%
[Etapa 2] mms_sine:M1_EXACT        | dev=cpu | NRMSE=0.014398451688533433 | dTmax=0.0048886320342518275 K | bal=0.7766921198303385%
[Etapa 2] mms_sine:M0R             | dev=cpu | NRMSE=0.8742310662304658 | dTmax=0.39939317915339245 K | bal=0.11741377859794883%
[Etapa 2] mms_sine:CUSPR           | dev=cpu | NRMSE=0.8742310662304658 | dTmax=0.39939317915339245 K | bal=0.11741377859794883%
[Etapa 2] cable_disk:M1            | dev=cpu | NRMSE=26.967034955671 | dTmax=117.82624049699967 K | bal=5.081450851136057%
[Etapa 2] cable_disk:M2            | dev=cpu | NRMSE=27.10169888250188 | dTmax=118.45224566972917 K | 

WindowsPath('C:/usr/ths_mia_fiis/tesis_MIA_2026-2_pinn/notebooks_campaign/campaign_workspace/stage2_selection_summary.json')

## Lógica de Screening y Promoción de Modelos

In [4]:
# Agregación multi-caso por modelo y regla de poda
per_model = {}
for rec in stage2_summary.values():
    if rec.get("status") != "completed":
        continue
    arch = rec["architecture"]
    row = per_model.setdefault(arch, {"scores": [], "nrmse": [], "balance": [], "dtmax": [], "elapsed": []})
    bal = rec["balance_pct"] if rec["balance_pct"] is not None else 100.0
    nrmse = rec["nrmse_pct"] if rec["nrmse_pct"] is not None else 100.0
    dtmax = rec["error_Tmax_abs_K"] if rec["error_Tmax_abs_K"] is not None else 100.0
    # Score relativo a umbrales estrictos (menor = mejor)
    score = (dtmax / STRICT_CRITERIA["error_Tmax_abs_K"]) + (nrmse / STRICT_CRITERIA["nrmse_pct"]) + (bal / STRICT_CRITERIA["balance_pct"])
    row["scores"].append(score)
    row["nrmse"].append(nrmse)
    row["balance"].append(bal)
    row["dtmax"].append(dtmax)
    row["elapsed"].append(rec.get("elapsed_s") or 0.0)

ranked = []
for arch, vals in per_model.items():
    if len(vals["scores"]) == len(CASES_STAGE2):
        ranked.append({
            "architecture": arch,
            "mean_score": float(np.mean(vals["scores"])),
            "max_balance_pct": float(np.max(vals["balance"])),
            "mean_nrmse_pct": float(np.mean(vals["nrmse"])),
            "total_elapsed_s": float(np.sum(vals["elapsed"])),
        })
ranked.sort(key=lambda r: r["mean_score"])

# Retenidos oficiales de la tesis (FINAL_CAMPAIGN_2026-10-27) combinados con ranking dinámico
THESIS_RETAINED = ["M1", "M1_EXACT", "CUSPR", "M0R", "M2"]
dynamic_top = [r["architecture"] for r in ranked[:5]]
promoted_models = [m for m in THESIS_RETAINED if m in per_model] or dynamic_top
finalists = promoted_models[:2]  # Top 1-2 para corridas profundas de 8000 pasos

payload = {
    "run_mode": RUN_MODE,
    "ranking": ranked,
    "promoted_models": promoted_models,
    "long_run_finalists": finalists,
}
write_json_atomic(CAMP_WS / "promoted_models.json", payload)
print("Ranking screening:", json.dumps(ranked, indent=2))
print("Modelos promovidos a Etapa 3/4:", promoted_models)

Ranking screening: [
  {
    "architecture": "M1_EXACT",
    "mean_score": 1238.7794418729184,
    "max_balance_pct": 0.8202970589856885,
    "mean_nrmse_pct": 13.552805647876808,
    "total_elapsed_s": 138.59963700012304
  },
  {
    "architecture": "M1",
    "mean_score": 1246.5264342367927,
    "max_balance_pct": 5.081450851136057,
    "mean_nrmse_pct": 14.317780032210614,
    "total_elapsed_s": 96.5882918999996
  },
  {
    "architecture": "M0R",
    "mean_score": 1341.5266337879664,
    "max_balance_pct": 103.78705293013881,
    "mean_nrmse_pct": 13.934955809405208,
    "total_elapsed_s": 116.32009059994016
  },
  {
    "architecture": "CUSPR",
    "mean_score": 1376.6214398499192,
    "max_balance_pct": 122.75836398766847,
    "mean_nrmse_pct": 14.10315770839921,
    "total_elapsed_s": 129.667719900026
  },
  {
    "architecture": "M2",
    "mean_score": 1464.458896103861,
    "max_balance_pct": 52.32758904603789,
    "mean_nrmse_pct": 25.965191280054313,
    "total_elapsed_s": 9